In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CrackSegDiff: Simplified Training & Inference (v3.0 - Fixed Benchmark)
Automated setup, data preparation (First 500 Test / 2000 Train), training, and testing.

In [ ]:
# 1. Setup Environment & Weights
# @title Configuration
TRAIN_MODEL = True # @param {type:"boolean"}
DRIVE_MODEL_DIR = "/content/drive/MyDrive/CrackSegDiff/models"

import os
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

!nvidia-smi
import os
if not os.path.exists('CrackSegDiff'):
    !git clone https://github.com/Ludwig-H/CrackSegDiff.git
%cd CrackSegDiff
!git pull

target_file_gd = 'CrackSegDiff/guided_diffusion/gaussian_diffusion.py'
if os.path.exists(target_file_gd):
    with open(target_file_gd, 'r') as f: content = f.read()
    content = content.replace('beta_end = scale * 0.02', 'beta_end = scale * 0.02\n        if beta_end > 0.999: beta_end = 0.999')
    with open(target_file_gd, 'w') as f: f.write(content)

# Downgrade PyTorch to stable 2.4.0 for guaranteed Mamba compatibility
print("Installing PyTorch 2.4.0 compatible with Mamba wheels...")
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

!pip install -r requirement.txt

# Force install pre-built wheels for Mamba-SSM and Causal-Conv1d to avoid compilation errors and ensure GPU speed
print("Installing Optimized Mamba Kernels...")
import torch
cuda_version = torch.version.cuda.replace('.', '')
torch_version = torch.__version__.split('+')[0].replace('.', '')
# Assuming standard Colab PyTorch 2.x and CUDA 11.8 or 12.x
# We use the releases from Dao-AILab which are reliable
!pip install ninja  # Speeds up compilation
!pip install causal-conv1d>=1.0.0 --no-build-isolation -v
!pip install mamba-ssm>=1.0.1 --no-build-isolation -v

# If the above standard install fails (it tries to build), we could try finding wheels:
# (But usually --no-build-isolation helps or just standard pip works if env is clean)

# Verify installation
try:
    import mamba_ssm
    print("SUCCESS: Mamba SSM installed successfully! GPU acceleration enabled (x10 speed).")
except ImportError:
    print("WARNING: Mamba SSM installation failed.")
    print("Fallback to Pure Python active (SLOWER).")

# Download Pretrained Weights
!mkdir -p pretrained_weights
!gdown 1JYqMxM5dbCLZ-WGPKtIofYJhj0VPuy3l -O pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth

# Patch Hardcoded Paths
target_file = 'CrackSegDiff/guided_diffusion/unet.py'
new_path = os.path.abspath('pretrained_weights/vssm_base_0229_ckpt_epoch_237.pth')
if os.path.exists(target_file):
    with open(target_file, 'r') as f: content = f.read()
    content = content.replace('/home/dell/jlc/segdiff/pre_trained_weights/vssm_base_0229_ckpt_epoch_237.pth', new_path)
    with open(target_file, 'w') as f: f.write(content)
    print("Path patched successfully.")

In [ ]:
# 2. Prepare Data (First 500 Test / 2000 Train)
!rm -rf data && mkdir -p data
%cd data
!gdown 1qnLMCeon7LJjT9H0ENiNF5sFs-F7-NvK -O data.zip
!unzip -q -o data.zip
%cd ..

import os, glob, shutil
print("Organizing Data...")

# Find folders
try:
    src_img = glob.glob('data/**/img/fused', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]
except IndexError:
    # Fallback if 'fused' not found, try 'intensity'
    print("Fused folder not found, checking intensity...")
    src_img = glob.glob('data/**/img/intensity', recursive=True)[0]
    src_lbl = glob.glob('data/**/lbs', recursive=True)[0]

print(f"Images source: {src_img}")
print(f"Labels source: {src_lbl}")

# Get sorted file lists
img_files = sorted(glob.glob(os.path.join(src_img, '*')))
lbl_files = sorted(glob.glob(os.path.join(src_lbl, '*.bmp')))

# Split: First 500 Test, Rest Train
test_pairs = list(zip(img_files[:500], lbl_files[:500]))
train_pairs = list(zip(img_files[500:], lbl_files[500:]))

print(f"Test Set: {len(test_pairs)} (First 500)")
print(f"Train Set: {len(train_pairs)} (Rest)")

# Copy to formatted directories
train_dir = os.path.abspath('data/Train')
test_dir = os.path.abspath('data/Test')

for pairs, dest in [(train_pairs, train_dir), (test_pairs, test_dir)]:
    os.makedirs(os.path.join(dest, '5d'), exist_ok=True)
    os.makedirs(os.path.join(dest, 'mask'), exist_ok=True)
    for img, mask in pairs:
        shutil.copy(img, os.path.join(dest, '5d'))
        shutil.copy(mask, os.path.join(dest, 'mask'))
print("Data preparation complete.")

In [ ]:
# @title 🔍 DIAGNOSTIC: Check Training Data Format
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("--- DIAGNOSTIC START ---")

# 1. Locate the unzipped data (from previous cell)
try:
    src_img_fused = glob.glob('data/**/img/fused', recursive=True)[0]
    print(f"Found fused images folder: {src_img_fused}")

    # 2. Load a sample
    # Updated to look for TIF and TIFF files
    sample_files = sorted(glob.glob(os.path.join(src_img_fused, '*.tif')) + glob.glob(os.path.join(src_img_fused, '*.tiff')) + glob.glob(os.path.join(src_img_fused, '*.png')))
    if not sample_files:
        print("No image files (tif/tiff/png) found in fused folder.")
    else:
        sample_path = sample_files[0]
        print(f"Analyzing sample: {sample_path}")

        try:
            img = Image.open(sample_path)
            print(f"Format: {img.format}, Mode: {img.mode}, Size: {img.size}")
            arr = np.array(img)
        except Exception as e_pil:
            print(f"PIL failed: {e_pil}. Trying tifffile...")
            import tifffile
            arr = tifffile.imread(sample_path)
            print("tifffile loaded successfully.")
        print(f"Shape: {arr.shape}, Dtype: {arr.dtype}")
        print(f"Min: {arr.min()}, Max: {arr.max()}")

        if arr.ndim == 3:
            # Multichannel
            n_ch = arr.shape[2]
            print(f"Channels analysis ({n_ch} channels):")
            channels = ['Ch0 (Int?)', 'Ch1 (Range?)', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6']
            for i in range(min(n_ch, len(channels))):
                c_name = channels[i]
                ch = arr[..., i]
                print(f"  {c_name}: Min={ch.min()}, Max={ch.max()}, Mean={ch.mean():.2f}")

            # Visualization
            # Display RGB preview (first 3 channels) + individual channels
            n_cols = min(n_ch, 6) + 1
            fig, axes = plt.subplots(1, n_cols, figsize=(4*n_cols, 4))
            
            # RGB Preview
            if n_ch >= 3:
                axes[0].imshow(arr[..., :3])
                axes[0].set_title("RGB Preview (0,1,2)")
            else:
                axes[0].imshow(arr[..., 0], cmap='gray')
                axes[0].set_title("Grayscale Preview")
            
            # Individual Channels
            for i in range(min(n_ch, 6)):
                axes[i+1].imshow(arr[..., i], cmap='gray')
                axes[i+1].set_title(f"Channel {i}")
            plt.show()
        else:
            print("Image is 2D (Grayscale). This implies Single Modality or Palette?")
            plt.imshow(arr, cmap='gray')
            plt.show()

except IndexError:
    print("Could not locate 'img/fused' folder. Ensure the previous cell (Data Preparation) ran successfully.")
except Exception as e:
    print(f"An error occurred: {e}")

print("--- DIAGNOSTIC END ---")

In [ ]:
# @title ⚖️ COMPARISON: Original vs Reconstructed Noisy (im00002)
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
import tifffile

id_img = 'im00002'

# 1. Auto-locate Original Fused TIF
possible_paths = [
    f'data/data/img/fused/{id_img}.tif',
    f'data/img/fused/{id_img}.tif',
    f'data/data/img/fused/{id_img}.tiff'
]
path_orig = None
for p in possible_paths:
    if os.path.exists(p):
        path_orig = p
        break

# 2. Noisy Source Files (Level 0.0 - Baseline)
path_noisy_int = f'/content/drive/MyDrive/Datasets/FIND/Noisy/both/0p0000/{id_img}_intensity.png'
path_noisy_rng = f'/content/drive/MyDrive/Datasets/FIND/Noisy/both/0p0000/{id_img}_range.png'

print(f"--- COMPARISON FOR {id_img} ---")

if not path_orig:
    print(f"ERROR: Original file {id_img} not found in expected paths. Check Data Preparation.")
elif not os.path.exists(path_noisy_int) or not os.path.exists(path_noisy_rng):
    print(f"ERROR: Noisy files not found on Drive. Check paths.")
else:
    # --- LOAD ORIGINAL ---
    try:
        orig = tifffile.imread(path_orig)
    except Exception as e:
        print(f"tifffile failed ({e}), trying skimage...")
        orig = io.imread(path_orig)
    print(f"Original Shape: {orig.shape}, Dtype: {orig.dtype}, Range: [{orig.min()}, {orig.max()}]")

    # --- LOAD & RECONSTRUCT NOISY ---
    try:
        n_int = io.imread(path_noisy_int)
        n_rng = io.imread(path_noisy_rng)
        print(f"Noisy Int Shape: {n_int.shape}, Noisy Range Shape: {n_rng.shape}")

        # Simulation of Benchmark Reconstruction
        H, W = n_int.shape[:2]
        recon = np.zeros((H, W, 3), dtype=np.uint8)
        
        # Channel 0: Intensity
        val_int = n_int if n_int.ndim==2 else n_int[...,0]
        recon[..., 0] = val_int
        
        # Channel 1: Range (Taking Red Channel from JET as done in benchmark)
        val_rng = n_rng if n_rng.ndim==2 else n_rng[...,0]
        recon[..., 1] = val_rng

        # --- COMPARISON ---
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # Row 1: Original Channels 0-3
        for i in range(4):
            if orig.ndim == 3 and orig.shape[2] > i:
                axes[0,i].imshow(orig[...,i], cmap='gray')
                axes[0,i].set_title(f'Orig Ch{i}')
            else:
                axes[0,i].text(0.5, 0.5, 'N/A', ha='center')
        
        # Row 2: Noisy Sources (Candidate Recons)
        axes[1,0].imshow(val_int, cmap='gray'); axes[1,0].set_title('Noisy Int (Source)')
        if n_rng.ndim == 3:
            axes[1,1].imshow(n_rng[...,0], cmap='gray'); axes[1,1].set_title('Noisy Range R')
            axes[1,2].imshow(n_rng[...,1], cmap='gray'); axes[1,2].set_title('Noisy Range G')
            axes[1,3].imshow(n_rng[...,2], cmap='gray'); axes[1,3].set_title('Noisy Range B')
        else:
            axes[1,1].text(0.5, 0.5, 'Range is not RGB', ha='center')

        # Row 3: Quantification (NMAE)
        if orig.ndim == 3 and orig.shape[2] >= 4 and n_rng.ndim == 3:
            def calc_nmae(a, b):
                a_n = a.astype(float) / (a.max() + 1e-8)
                b_n = b.astype(float) / (b.max() + 1e-8)
                return np.abs(a_n - b_n).mean()

            err_int = calc_nmae(orig[...,0], val_int)
            err_r = calc_nmae(orig[...,1], n_rng[...,0])
            err_g = calc_nmae(orig[...,2], n_rng[...,1])
            err_b = calc_nmae(orig[...,3], n_rng[...,2])
            
            axes[2,0].text(0.5, 0.5, f"Int NMAE:\n{err_int:.4f}", ha='center', fontsize=14)
            axes[2,1].text(0.5, 0.5, f"Range R vs Orig Ch1:\n{err_r:.4f}", ha='center', fontsize=14)
            axes[2,2].text(0.5, 0.5, f"Range G vs Orig Ch2:\n{err_g:.4f}", ha='center', fontsize=14)
            axes[2,3].text(0.5, 0.5, f"Range B vs Orig Ch3:\n{err_b:.4f}", ha='center', fontsize=14)
            
            avg_jet_err = (err_r + err_g + err_b) / 3
            print(f"--- HYPOTHESIS RESULTS ---")
            print(f"Match Int: {err_int:.4f}")
            print(f"Match Jet (Avg R,G,B): {avg_jet_err:.4f}")
            
            if avg_jet_err < 0.15:
                print("✅ HYPOTHESIS CONFIRMED: Original data IS [Int, JetR, JetG, JetB].")
            else:
                print("❌ HYPOTHESIS FAILED: Original data structure is different.")
            
            # --- HYPOTHESIS 2 CHECK: [JetR, JetG, JetB, Int] ---
            err_r2 = calc_nmae(orig[...,0], n_rng[...,0])
            err_g2 = calc_nmae(orig[...,1], n_rng[...,1])
            err_b2 = calc_nmae(orig[...,2], n_rng[...,2])
            err_int2 = calc_nmae(orig[...,3], val_int)
            
            avg_jet_err2 = (err_r2 + err_g2 + err_b2) / 3
            print(f"\n--- HYPOTHESIS 2 CHECK: [JetR, JetG, JetB, Int] ---")
            print(f"Match Jet (Avg): {avg_jet_err2:.4f}")
            print(f"Match Int: {err_int2:.4f}")
            
            if avg_jet_err2 < 0.15 and err_int2 < 0.15:
                 print("✅ HYPOTHESIS 2 CONFIRMED: Data is [JetR, JetG, JetB, Int].")
            
            # --- HYPOTHESIS 3 CHECK: [JetR, JetG, JetB, Int, Int, Int] ---
            if orig.shape[2] >= 6:
                err_int3 = calc_nmae(orig[...,4], val_int)
                err_int4 = calc_nmae(orig[...,5], val_int)
                
                print(f"\n--- HYPOTHESIS 3 CHECK: [JetR, JetG, JetB, Int, Int, Int] ---")
                print(f"Match Int (Ch4): {err_int3:.4f}")
                print(f"Match Int (Ch5): {err_int4:.4f}")
                
                if avg_jet_err2 < 0.15 and err_int2 < 0.15 and err_int3 < 0.15 and err_int4 < 0.15:
                     print("✅ ✅ HYPOTHESIS 3 CONFIRMED (PERFECT MATCH): Data is [JetR, JetG, JetB, Int, Int, Int].")
                else:
                     print("❌ HYPOTHESIS 3 FAILED: Channels 4/5 do not match Intensity.")
        else:
             axes[2,0].text(0.5, 0.5, "Cannot compute diff (dims mismatch)", ha='center')

        for ax in axes.flatten(): ax.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"An error occurred during comparison: {e}")

In [ ]:
# 3. Train Model
import os
import shutil
import glob

data_dir = os.path.abspath('data/Train')
out_dir = os.path.abspath('results/train_output')
os.makedirs(out_dir, exist_ok=True)

if TRAIN_MODEL:
    print("Starting Training...")
    !python CrackSegDiff/segmentation_train.py --data_dir {data_dir} --out_dir {out_dir} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --lr 5e-5 --batch_size 8 --save_interval 5000 --lr_anneal_steps 40000

    # Save to Drive
    print(f"Backing up model to {DRIVE_MODEL_DIR}...")
    for model_file in glob.glob(os.path.join(out_dir, '*.pt')):
        shutil.copy(model_file, DRIVE_MODEL_DIR)
    print("Backup complete.")
else:
    print("Training skipped (TRAIN_MODEL=False).")

In [ ]:
# 4. Inference
import glob, os

# Select Model
if TRAIN_MODEL:
    models = sorted(glob.glob('results/train_output/*.pt'))
    model_path = models[-1] if models else "pretrained_weights/savedmodel100000.pt"
else:
    print(f"Using model from Drive: {DRIVE_MODEL_DIR}")
    drive_models = sorted(glob.glob(os.path.join(DRIVE_MODEL_DIR, '*.pt')))
    model_path = drive_models[-1] if drive_models else "pretrained_weights/savedmodel100000.pt"

test_dir = os.path.abspath('data/Test')
print(f"Using model: {model_path}")

for modality in ['fused']: # ['intensity', 'range', 'fused']
    out_path = f"results/test_output_{modality}"
    os.makedirs(out_path, exist_ok=True)
    print(f"Testing {modality}...")
    !python CrackSegDiff/segmentation_sample.py --data_dir {test_dir} --out_dir {out_path} --model_path {model_path} --modality {modality} --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1
    print(f"Done {modality}")

In [ ]:
# 5. Zip Results
!zip -r results.zip results

In [ ]:
# 6. Robustness Benchmark (Inference on Pre-generated Noisy Data)
# Uses noisy data from /content/drive/MyDrive/Datasets/FIND/Noisy
# Saves results to /content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise

import numpy as np
import os
import glob
import shutil
from skimage import io
from tqdm import tqdm

# --- Configuration --- 
NOISY_INPUT_ROOT = "/content/drive/MyDrive/Datasets/FIND/Noisy"
RESULTS_ROOT = "/content/drive/MyDrive/Datasets/FIND/Results/CrackSegDiff_noise"
TEMP_DATA_ROOT = os.path.abspath("temp_inference_noisy")
CLEAN_TEST_DIR = os.path.abspath('data/Test') # Source for masks

# Select Model (Same logic as above)
if 'TRAIN_MODEL' in locals() and not TRAIN_MODEL:
    drive_models = sorted(glob.glob(os.path.join(DRIVE_MODEL_DIR, '*.pt')))
    MODEL_PATH = drive_models[-1] if drive_models else "pretrained_weights/savedmodel100000.pt"
else:
    models = sorted(glob.glob('results/train_output/*.pt'))
    MODEL_PATH = models[-1] if models else "pretrained_weights/savedmodel100000.pt"

print(f"Using Model for Benchmark: {MODEL_PATH}")

speckle_vars = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]
range_sigmas = [0.0, 0.01, 0.05, 0.10, 0.3, 0.5]

experiments = [
    # ("speckle_intensity", speckle_vars),
    # ("gauss_range", range_sigmas),
    ("both", range_sigmas)
]
def _noise_tag(x: float, ndigits: int = 4):
    return f"{x:.{ndigits}f}".replace(".", "p")

# Ensure we have masks ready
clean_masks = sorted(glob.glob(os.path.join(CLEAN_TEST_DIR, 'mask', '*.bmp')))
clean_masks = clean_masks[:500]
if not clean_masks:
    print("WARNING: No masks found in clean test dir. Masks will be missing in temp dir.")

for exp_name, levels in experiments:
    print(f"\n=== Experiment: {exp_name} ===")

    for lvl in levels:
        lvl = float(lvl)
        tag = _noise_tag(lvl)

        # Input Directory (Drive)
        noisy_src_dir = os.path.join(NOISY_INPUT_ROOT, exp_name, tag)
        # Output Directory (Drive)
        final_dest_dir = os.path.join(RESULTS_ROOT, exp_name, tag, "test_output_fused")

        if not os.path.exists(noisy_src_dir):
            print(f"Skipping {exp_name}/{tag}: Input directory not found ({noisy_src_dir})")
            continue

        if os.path.exists(final_dest_dir) and len(glob.glob(os.path.join(final_dest_dir, '*.png'))) >= 500:
            print(f"Skipping {exp_name}/{tag}: Results already exist ({len(os.listdir(final_dest_dir)) } files)")
            continue

        print(f"Processing {exp_name} - Level {lvl} (Tag: {tag})")

        # 1. Prepare Temp Directory
        if os.path.exists(TEMP_DATA_ROOT): shutil.rmtree(TEMP_DATA_ROOT)
        temp_5d = os.path.join(TEMP_DATA_ROOT, '5d')
        temp_mask = os.path.join(TEMP_DATA_ROOT, 'mask')
        os.makedirs(temp_5d, exist_ok=True)
        os.makedirs(temp_mask, exist_ok=True)

        # 2. Reconstruct Fused Images from Drive (Intensity + Range)
        # Files are named imXXXXX_intensity.png / imXXXXX_range.png
        int_files = sorted(glob.glob(os.path.join(noisy_src_dir, '*_intensity.png')))
        print(f"Found {len(int_files)} intensity images in {noisy_src_dir}")

        # Install tifffile for robust 6-channel saving
        try: import tifffile
        except ImportError: 
            !pip install tifffile
            import tifffile

        count = 0
        for int_path in int_files:
            fname = os.path.basename(int_path)
            # fname format: imXXXXX_intensity.png
            base_name = fname.replace('_intensity.png', '') # imXXXXX
            range_path = os.path.join(noisy_src_dir, f"{base_name}_range.png")

            if not os.path.exists(range_path):
                continue

            # Read Images
            img_i = io.imread(int_path)
            img_r = io.imread(range_path)

            # Ensure dimensions match
            if img_i.shape[:2] != img_r.shape[:2]:
                continue

            # CORRECT RECONSTRUCTION (Hypothesis 2: RangeJet + Int + Padding)
            H, W = img_i.shape[:2]
            fused = np.zeros((H, W, 6), dtype=np.uint8)

            # 1. Range (Jet RGB) -> Channels 0,1,2
            if img_r.ndim == 3 and img_r.shape[2] >= 3:
                fused[..., 0] = img_r[..., 0]
                fused[..., 1] = img_r[..., 1]
                fused[..., 2] = img_r[..., 2]
            else:
                # Fallback grayscale
                fused[..., 0] = img_r if img_r.ndim==2 else img_r[...,0]

            # 2. Intensity -> Channels 3, 4, 5 (Triplicated Intensity)
            val_int = img_i if img_i.ndim==2 else img_i[...,0]
            fused[..., 3] = val_int
            fused[..., 4] = val_int
            fused[..., 5] = val_int

            # Save as TIF (6 channels)
            tifffile.imwrite(os.path.join(temp_5d, f"{base_name}.tif"), fused)
            count += 1

        # Copy Masks (Assumes matching IDs in Clean Test Dir)
        # We blindly copy all first 500 clean masks, assuming IDs match 1-500.
        for mask_path in clean_masks:
            shutil.copy(mask_path, temp_mask)

        print(f"Prepared {count} images for inference.")

        # 3. Run Inference
        temp_out_dir = os.path.join(TEMP_DATA_ROOT, "output")
        os.makedirs(temp_out_dir, exist_ok=True)

        !python CrackSegDiff/segmentation_sample.py --data_dir {TEMP_DATA_ROOT} --out_dir {temp_out_dir} --model_path {MODEL_PATH} --modality all --image_size 256 --num_channels 96 --class_cond False --num_res_blocks 2 --num_heads 1 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16 --diffusion_steps 500 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --num_ensemble 1

        # 4. Save to Drive
        print(f"Saving results to: {final_dest_dir}")
        os.makedirs(final_dest_dir, exist_ok=True)

        copied = 0
        for png in glob.glob(os.path.join(temp_out_dir, "*.png")):
            shutil.copy(png, final_dest_dir)
            copied += 1
        print(f"Saved {copied} result files.")

print("Robustness Benchmark (Inference Only) Complete.")